# N3 — Escenarios metafóricos (Visualización)
## AI-MELT: Nivel 3 — Exploración visual

**Prerequisito:** Haber ejecutado `N3_escenarios.ipynb` (procesamiento).

---

### Archivos de entrada esperados (en `./outputs/N3/`)

| Archivo | Columnas clave | Descripción |
|---|---|---|
| `n3_escenarios.csv` | ID_escenario, ID_metaconvencional_base, nombre_escenario, estatus, valoracion_uso, sentimiento_dominante_validacion, emocion_dominante_validacion | Un escenario por fila |
| `n3_secuencia_narrativa.csv` | ID_secuencia, ID_escenario, acto (inicio/desarrollo/desenlace), descripcion | Tres actos por escenario |
| `n3_posicionamiento.csv` | ID_posicionamiento, ID_escenario, grupo_social, accion_atribuida, legitimacion (legitimada/deslegitimada) | Actores sociales por escenario |
| `n3_sesgo_evaluativo.csv` | ID_sesgo, ID_escenario, polo_positivo, polo_negativo, sentimiento_pos/neg/neu_validacion | Polos evaluativos + validación |
| `n3_afecto.csv` | ID_afecto, ID_escenario, emocion, tipo (facilitado/inhibido), funcion_social, marcadores_linguisticos | Emociones por escenario |

### Archivos de N2 también usados

| Archivo | Columnas clave |
|---|---|
| `n2_metaforas_convencionales.csv` | ID_metaconvencional, metafora_conceptual, frecuencia_absoluta, dispersion |

---

### Visualizaciones generadas
1. Tabla resumen de escenarios
2. Radar chart por escenario (frecuencia, distribución, polaridad, nº actores)
3. Red de actores por escenario
4. Barras agrupadas: afectos facilitados vs. inhibidos
5. Validación: sentimiento LLM vs. pysentimiento

## 1. Configuración y carga de datos

In [ ]:
# ============================================================
# 1. IMPORTS Y CARGA
# ============================================================
# Descomenta si necesitas instalar:
# !pip install matplotlib seaborn plotly networkx pandas

import gc
import os
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns

warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

N2_DIR = "./outputs/N2/"
N3_DIR = "./outputs/N3/"
VIZ_DIR = "./outputs/N3/viz/"
os.makedirs(VIZ_DIR, exist_ok=True)

# Cargar datos
df_esc = pd.read_csv(os.path.join(N3_DIR, "n3_escenarios.csv"))
df_narr = pd.read_csv(os.path.join(N3_DIR, "n3_secuencia_narrativa.csv"))
df_pos = pd.read_csv(os.path.join(N3_DIR, "n3_posicionamiento.csv"))
df_sesgo = pd.read_csv(os.path.join(N3_DIR, "n3_sesgo_evaluativo.csv"))
df_afecto = pd.read_csv(os.path.join(N3_DIR, "n3_afecto.csv"))
df_conv = pd.read_csv(os.path.join(N2_DIR, "n2_metaforas_convencionales.csv"))

print("✓ Datos cargados:")
print(f"  Escenarios: {len(df_esc)}")
print(f"  Secuencias narrativas: {len(df_narr)}")
print(f"  Posicionamientos: {len(df_pos)}")
print(f"  Sesgos evaluativos: {len(df_sesgo)}")
print(f"  Afectos: {len(df_afecto)}")

## 2. Tabla resumen de escenarios

In [ ]:
# ============================================================
# 2. TABLA RESUMEN INTERACTIVA (Plotly)
# ============================================================

# Enriquecer con datos de N2
df_esc_enriched = df_esc.merge(
    df_conv[["ID_metaconvencional", "frecuencia_absoluta", "dispersion", "robustez"]].drop_duplicates(),
    left_on="ID_metaconvencional_base", right_on="ID_metaconvencional", how="left"
)

# Contar actores y afectos por escenario
n_actores = df_pos.groupby("ID_escenario").size().to_dict()
n_afectos = df_afecto.groupby("ID_escenario").size().to_dict()
df_esc_enriched["n_actores"] = df_esc_enriched["ID_escenario"].map(n_actores).fillna(0).astype(int)
df_esc_enriched["n_afectos"] = df_esc_enriched["ID_escenario"].map(n_afectos).fillna(0).astype(int)

# Tabla Plotly
cols_display = ["nombre_escenario", "estatus", "valoracion_uso", "frecuencia_absoluta",
                "n_actores", "n_afectos", "sentimiento_dominante_validacion"]
df_display = df_esc_enriched[cols_display].copy()
df_display.columns = ["Escenario", "Estatus", "Valoración", "Frecuencia", "Actores", "Afectos", "Sentimiento val."]

# Colorear por estatus
color_estatus = {"DOMINANTE": "#1D9E75", "CHALLENGER": "#E8A838", "EMERGENTE": "#3B8BD4", "PERIFÉRICO": "#999999"}
cell_colors = [[color_estatus.get(e, "#ffffff") for e in df_display["Estatus"]]]

fig = go.Figure(data=[go.Table(
    header=dict(values=list(df_display.columns), fill_color='#1F4E79',
                font=dict(color='white', size=11), align='left'),
    cells=dict(values=[df_display[c] for c in df_display.columns],
               fill_color=['white'] * len(df_display.columns),
               font=dict(size=10), align='left', height=25)
)])
fig.update_layout(title="Escenarios metafóricos — Resumen", height=max(400, 30*len(df_display)))
out_path = os.path.join(VIZ_DIR, "viz_tabla_escenarios.html")
pio.write_html(fig, out_path, include_plotlyjs="cdn")
print(f"✓ {out_path}")
fig.show()

## 3. Radar chart por escenario

Dimensiones: frecuencia, dispersión, nº actores, nº afectos. Se muestran los escenarios dominantes y challengers.

In [ ]:
# ============================================================
# 3. RADAR CHART POR ESCENARIO
# ============================================================

top_escenarios = df_esc_enriched[df_esc_enriched["estatus"].isin(["DOMINANTE", "CHALLENGER"])].head(8)

if len(top_escenarios) > 0:
    categories = ["Frecuencia", "Dispersión", "Actores", "Afectos"]

    fig = go.Figure()
    colors = ["#1D9E75", "#3B8BD4", "#D85A30", "#7F77DD", "#E8A838", "#FF6B8A", "#44B89D", "#9B59B6"]

    for idx, (_, row) in enumerate(top_escenarios.iterrows()):
        # Normalizar valores a [0, 1]
        max_freq = df_esc_enriched["frecuencia_absoluta"].max() or 1
        max_act = df_esc_enriched["n_actores"].max() or 1
        max_af = df_esc_enriched["n_afectos"].max() or 1

        values = [
            (row.get("frecuencia_absoluta", 0) or 0) / max_freq,
            row.get("dispersion", 0) or 0,
            (row.get("n_actores", 0) or 0) / max_act,
            (row.get("n_afectos", 0) or 0) / max_af,
        ]
        values.append(values[0])  # Cerrar el polígono

        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=categories + [categories[0]],
            fill='toself',
            name=str(row["nombre_escenario"])[:40],
            opacity=0.6,
            line=dict(color=colors[idx % len(colors)])
        ))

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        title="Radar: escenarios dominantes y challengers",
        height=600
    )
    out_path = os.path.join(VIZ_DIR, "viz_radar_escenarios.html")
    pio.write_html(fig, out_path, include_plotlyjs="cdn")
    print(f"✓ {out_path}")
    fig.show()
else:
    print("⚠ No hay escenarios dominantes/challengers para el radar chart.")

## 4. Red de actores por escenario

In [ ]:
# ============================================================
# 4. RED DE ACTORES POR ESCENARIO
# ============================================================
import networkx as nx

if len(df_pos) > 0:
    G = nx.Graph()

    # Nodos: escenarios + grupos sociales
    for _, row in df_esc.iterrows():
        G.add_node(row["ID_escenario"], type="escenario",
                    label=str(row["nombre_escenario"])[:30],
                    estatus=row.get("estatus", ""))

    for _, row in df_pos.iterrows():
        grupo = str(row["grupo_social"])
        if not G.has_node(grupo):
            G.add_node(grupo, type="actor", label=grupo)
        G.add_edge(row["ID_escenario"], grupo,
                     legitimacion=row.get("legitimacion", ""))

    # Visualizar con matplotlib
    fig, ax = plt.subplots(figsize=(16, 12))
    pos_layout = nx.spring_layout(G, seed=42, k=2)

    # Dibujar nodos
    esc_nodes = [n for n, d in G.nodes(data=True) if d.get("type") == "escenario"]
    act_nodes = [n for n, d in G.nodes(data=True) if d.get("type") == "actor"]

    nx.draw_networkx_nodes(G, pos_layout, nodelist=esc_nodes, node_color="#3B8BD4",
                            node_size=400, alpha=0.8, ax=ax)
    nx.draw_networkx_nodes(G, pos_layout, nodelist=act_nodes, node_color="#D85A30",
                            node_size=250, alpha=0.8, ax=ax)

    # Aristas coloreadas por legitimación
    leg_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get("legitimacion") == "legitimada"]
    deleg_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get("legitimacion") == "deslegitimada"]
    other_edges = [(u, v) for u, v, d in G.edges(data=True)
                    if d.get("legitimacion") not in ["legitimada", "deslegitimada"]]

    nx.draw_networkx_edges(G, pos_layout, edgelist=leg_edges, edge_color="#1D9E75",
                            alpha=0.6, width=1.5, ax=ax)
    nx.draw_networkx_edges(G, pos_layout, edgelist=deleg_edges, edge_color="#D85A30",
                            alpha=0.6, width=1.5, style="dashed", ax=ax)
    nx.draw_networkx_edges(G, pos_layout, edgelist=other_edges, edge_color="#999",
                            alpha=0.4, width=1, ax=ax)

    # Labels
    labels = {n: d.get("label", n)[:20] for n, d in G.nodes(data=True)}
    nx.draw_networkx_labels(G, pos_layout, labels, font_size=7, ax=ax)

    ax.set_title("Red de actores por escenario (azul=escenario, naranja=actor, verde=legitimado, rojo=deslegitimado)")
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, "viz_red_actores.png"), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    del G
else:
    print("⚠ No hay datos de posicionamiento para la red de actores.")

gc.collect()

## 5. Afectos facilitados vs. inhibidos

In [ ]:
# ============================================================
# 5. BARRAS AGRUPADAS: AFECTOS FACILITADOS vs INHIBIDOS
# ============================================================

if len(df_afecto) > 0:
    # Contar emociones por tipo
    afecto_counts = df_afecto.groupby(["emocion", "tipo"]).size().unstack(fill_value=0)

    # Asegurar que existan ambas columnas
    for col in ["facilitado", "inhibido"]:
        if col not in afecto_counts.columns:
            afecto_counts[col] = 0

    # Top 15 emociones por frecuencia total
    afecto_counts["total"] = afecto_counts.sum(axis=1)
    top_emociones = afecto_counts.sort_values("total", ascending=True).tail(15)

    fig, ax = plt.subplots(figsize=(12, 8))
    y_pos = range(len(top_emociones))

    ax.barh(y_pos, top_emociones.get("facilitado", 0), height=0.4,
             color="#1D9E75", label="Facilitado", align='center')
    ax.barh([y + 0.4 for y in y_pos], top_emociones.get("inhibido", 0), height=0.4,
             color="#D85A30", label="Inhibido", align='center')

    ax.set_yticks([y + 0.2 for y in y_pos])
    ax.set_yticklabels(top_emociones.index, fontsize=9)
    ax.set_xlabel("Número de escenarios")
    ax.set_title("Afectos facilitados vs. inhibidos (top 15 emociones)")
    ax.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, "viz_afectos.png"), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
else:
    print("⚠ No hay datos de afecto.")

## 6. Validación: sentimiento LLM vs. pysentimiento

In [ ]:
# ============================================================
# 6. VALIDACIÓN: LLM vs PYSENTIMIENTO
# ============================================================

if len(df_sesgo) > 0 and "sentimiento_pos_validacion" in df_sesgo.columns:
    df_val = df_esc.merge(df_sesgo[["ID_escenario", "sentimiento_pos_validacion",
                                      "sentimiento_neg_validacion"]], on="ID_escenario", how="left")

    # Comparar valoracion_uso del LLM con sentimiento pysentimiento
    df_val["sentimiento_pysentimiento"] = df_val.apply(
        lambda r: "POS" if (r.get("sentimiento_pos_validacion", 0) or 0) > (r.get("sentimiento_neg_validacion", 0) or 0)
                  else "NEG", axis=1)

    # Mapear valoracion_uso a POS/NEG
    val_map = {"POSITIVO": "POS", "NEGATIVO": "NEG", "NEUTRO": "NEU"}
    df_val["llm_polarity"] = df_val["valoracion_uso"].map(val_map).fillna("NEU")

    # Concordancia
    concordance = (df_val["llm_polarity"] == df_val["sentimiento_pysentimiento"]).mean()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Scatter: pos vs neg scores
    axes[0].scatter(df_val["sentimiento_pos_validacion"].fillna(0),
                     df_val["sentimiento_neg_validacion"].fillna(0),
                     c=df_val["llm_polarity"].map({"POS": "#1D9E75", "NEG": "#D85A30", "NEU": "#999"}),
                     s=60, alpha=0.7, edgecolors='white')
    axes[0].set_xlabel("Sentimiento POS (pysentimiento)")
    axes[0].set_ylabel("Sentimiento NEG (pysentimiento)")
    axes[0].set_title("Polaridad pysentimiento (color = LLM)")
    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)

    # Barras de concordancia
    concord_data = df_val.groupby(["llm_polarity", "sentimiento_pysentimiento"]).size().unstack(fill_value=0)
    concord_data.plot(kind="bar", ax=axes[1], color=["#D85A30", "#1D9E75"])
    axes[1].set_xlabel("Polaridad LLM")
    axes[1].set_ylabel("Conteo")
    axes[1].set_title(f"LLM vs pysentimiento (concordancia: {concordance:.1%})")
    axes[1].tick_params(axis='x', rotation=0)

    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, "viz_validacion_sentimiento.png"), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

    print(f"Concordancia LLM vs pysentimiento: {concordance:.1%}")
else:
    print("⚠ No hay datos de validación de sentimiento.")

## 7. Resumen

In [ ]:
# ============================================================
# 7. RESUMEN
# ============================================================

print("=" * 60)
print("N3 — VISUALIZACIONES COMPLETADAS")
print("=" * 60)
print(f"\nArchivos en {VIZ_DIR}:")
for f_name in sorted(os.listdir(VIZ_DIR)):
    size = os.path.getsize(os.path.join(VIZ_DIR, f_name)) / 1024
    icon = "📊" if f_name.endswith(".html") else "🖼"
    print(f"  {icon} {f_name} ({size:.0f} KB)")
print("\n➡ SIGUIENTE: Ejecutar N4_regimenes.ipynb")